# 初期モデル構築 - テーブルデータのみ

衛星データなしでテーブルデータのみを使用して、LightGBM と CatBoost で建設コスト（USD/m²）を予測します。
評価指標: RMSLE / K-Fold (k=3) クロスバリデーション

## 実験設定\n\n**`CASE_NAME` を変更すると対応する `conf/parameter_{CASE_NAME}.yml` が読み込まれる。**"

In [1]:
CASE_NAME = 'case002'

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns=200
sys.path.insert(0, '..')
from src.config import load_config
from src.features import TARGET, CAT_COLS, prepare_features
from src.training import rmsle, run_cv, make_lgb_fn, make_cb_fn
from src.io import save_submission

cfg = load_config(CASE_NAME)

TRAIN_CSV       = cfg['train_csv']
EVAL_CSV        = cfg['eval_csv']
SEED            = cfg['seed']
N_SPLITS        = cfg['n_splits']
EXTRA_DROP_COLS = cfg.get('extra_drop_cols')
LGB_PARAMS      = cfg['lgb_params']
CB_PARAMS       = cfg['cb_params']

RESULT_DIR = f'../data/result/{CASE_NAME}'
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Loaded: conf/parameter_{CASE_NAME}.yml')
print(f'Result directory: {RESULT_DIR}')

Loaded: conf/parameter_case002.yml
Result directory: ../data/result/case002


## データ読み込み

In [3]:
train_df = pd.read_csv(TRAIN_CSV)
eval_df  = pd.read_csv(EVAL_CSV)

print(f'Train shape: {train_df.shape}')
print(f'Eval  shape: {eval_df.shape}')
train_df.head(3)

Train shape: (1024, 348)
Eval  shape: (1024, 347)


,data_id,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,sentinel2_tiff_file_name,viirs_tiff_file_name,construction_cost_per_m2_usd,B1_pct0,B1_pct10,B1_pct20,B1_pct30,B1_pct40,B1_pct50,B1_pct60,B1_pct70,B1_pct80,B1_pct90,B1_pct100,B1_mean,B1_std,B2_pct0,B2_pct10,B2_pct20,B2_pct30,B2_pct40,B2_pct50,B2_pct60,B2_pct70,B2_pct80,B2_pct90,B2_pct100,B2_mean,B2_std,B3_pct0,B3_pct10,B3_pct20,B3_pct30,B3_pct40,B3_pct50,B3_pct60,B3_pct70,B3_pct80,B3_pct90,B3_pct100,B3_mean,B3_std,B4_pct0,B4_pct10,B4_pct20,B4_pct30,B4_pct40,B4_pct50,B4_pct60,B4_pct70,B4_pct80,B4_pct90,B4_pct100,B4_mean,B4_std,B5_pct0,B5_pct10,B5_pct20,B5_pct30,B5_pct40,B5_pct50,B5_pct60,B5_pct70,B5_pct80,B5_pct90,B5_pct100,B5_mean,B5_std,B6_pct0,B6_pct10,B6_pct20,B6_pct30,B6_pct40,B6_pct50,B6_pct60,B6_pct70,B6_pct80,B6_pct90,B6_pct100,B6_mean,...,BSI_pct40,BSI_pct50,BSI_pct60,BSI_pct70,BSI_pct80,BSI_pct90,BSI_pct100,BSI_mean,BSI_std,NBSI_pct0,NBSI_pct10,NBSI_pct20,NBSI_pct30,NBSI_pct40,NBSI_pct50,NBSI_pct60,NBSI_pct70,NBSI_pct80,NBSI_pct90,NBSI_pct100,NBSI_mean,NBSI_std,NDWI_pct0,NDWI_pct10,NDWI_pct20,NDWI_pct30,NDWI_pct40,NDWI_pct50,NDWI_pct60,NDWI_pct70,NDWI_pct80,NDWI_pct90,NDWI_pct100,NDWI_mean,NDWI_std,MNDWI_pct0,MNDWI_pct10,MNDWI_pct20,MNDWI_pct30,MNDWI_pct40,MNDWI_pct50,MNDWI_pct60,MNDWI_pct70,MNDWI_pct80,MNDWI_pct90,MNDWI_pct100,MNDWI_mean,MNDWI_std,Clay_pct0,Clay_pct10,Clay_pct20,Clay_pct30,Clay_pct40,Clay_pct50,Clay_pct60,Clay_pct70,Clay_pct80,Clay_pct90,Clay_pct100,Clay_mean,Clay_std,IronOxide_pct0,IronOxide_pct10,IronOxide_pct20,IronOxide_pct30,IronOxide_pct40,IronOxide_pct50,IronOxide_pct60,IronOxide_pct70,IronOxide_pct80,IronOxide_pct90,IronOxide_pct100,IronOxide_mean,IronOxide_std,Carbonate_pct0,Carbonate_pct10,Carbonate_pct20,Carbonate_pct30,Carbonate_pct40,Carbonate_pct50,Carbonate_pct60,Carbonate_pct70,Carbonate_pct80,Carbonate_pct90,Carbonate_pct100,Carbonate_mean,Carbonate_std,avg_rad_pct0,avg_rad_pct10,avg_rad_pct20,avg_rad_pct30,avg_rad_pct40,avg_rad_pct50,avg_rad_pct60,avg_rad_pct70,avg_rad_pct80,avg_rad_pct90,avg_rad_pct100,avg_rad_mean,avg_rad_std
0,LP81L,Dinagat Islands,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,sentinel_2_dinagat_islands_2019-Q3.tif,viirs_dinagat_islands_2019-Q3.tif,129.997420,79.0,235.0,282.0,318.0,360.0,411.0,491.0,676.0,1505.0,4305.9,12442.0,1306.214859,2083.542109,109.0,283.0,318.0,354.0,396.0,448.0,526.0,702.0,1390.0,4237.0,12850.0,1304.793386,2028.163855,112.0,492.0,544.0,595.0,653.0,724.0,814.0,973.0,1501.0,4131.0,12429.0,1476.873568,1876.246837,55.0,236.0,272.0,319.0,380.0,463.0,568.0,775.0,1406.0,3918.0,12107.0,1239.978003,1876.147938,66.0,799.0,881.0,956.0,1042.0,1147.0,1267.0,1438.0,1853.0,4381.0,12849.0,1836.360197,1858.186597,45.0,2110.0,2415.0,2593.0,2738.0,2872.0,3015.0,3193.0,3482.0,4747.0,12033.0,3184.346665,...,-0.318017,-0.259107,-0.195594,-0.138990,-0.096317,-0.051140,0.397685,-0.240089,0.152624,-0.696751,-0.452918,-0.427833,-0.402794,-0.366435,-0.313729,-0.255161,-0.200000,-0.147581,-0.084259,0.439789,-0.286398,0.153233,-0.850387,-0.750380,-0.726949,-0.701971,-0.670951,-0.633071,-0.586303,-0.486990,-0.261191,-0.037702,0.832858,-0.518670,0.267313,-0.738832,-0.507313,-0.477261,-0.452655,-0.428246,-0.401658,-0.367091,-0.283019,-0.096182,0.100802,0.863447,-0.303787,0.235211,0.607184,1.239295,1.393008,1.631015,1.863482,2.048593,2.204152,2.343933,2.459103,2.569671,3.558559,1.967181,0.498286,0.106383,0.747967,0.805690,0.853448,0.894484,0.928119,0.963603,1.018124,1.111111,1.268349,3.570909,0.981737,0.258615,0.169127,0.345718,0.368197,0.391037,0.426642,0.482034,0.549976,0.622843,0.696757,0.787294,2.256161,0.537562,0.201261,-0.04,0.11,0.13,0.15,0.16,0.17,0.1

## 特徴量エンジニアリング

In [4]:
X_train = prepare_features(train_df, extra_drop_cols=EXTRA_DROP_COLS)
y_train = train_df[TARGET].copy()

# eval CSV に衛星特徴量がない場合は None を指定して CV のみ実行
try:
    X_eval = prepare_features(eval_df, extra_drop_cols=EXTRA_DROP_COLS)
    if not set(X_train.columns) <= set(X_eval.columns):
        raise ValueError('eval に存在しない列があります。')
except (KeyError, ValueError) as e:
    print(f'WARNING: X_eval を None にします ({e})')
    X_eval = None

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_eval : {X_eval.shape if X_eval is not None else None}')
print('Features:', X_train.columns.tolist())

X_train: (1024, 344), y_train: (1024,)
X_eval : (1024, 344)
Features: ['geolocation_name', 'country', 'year', 'deflated_gdp_usd', 'us_cpi', 'developed_country', 'landlocked', 'region_economic_classification', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'straight_distance_to_capital_km', 'seismic_hazard_zone', 'flood_risk_class', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk', 'koppen_climate_zone', 'B1_pct0', 'B1_pct10', 'B1_pct20', 'B1_pct30', 'B1_pct40', 'B1_pct50', 'B1_pct60', 'B1_pct70', 'B1_pct80', 'B1_pct90', 'B1_pct100', 'B1_mean', 'B1_std', 'B2_pct0', 'B2_pct10', 'B2_pct20', 'B2_pct30', 'B2_pct40', 'B2_pct50', 'B2_pct60', 'B2_pct70', 'B2_pct80', 'B2_pct90', 'B2_pct100', 'B2_mean', 'B2_std', 'B3_pct0', 'B3_pct10', 'B3_pct20', 'B3_pct30', 'B3_pct40', 'B3_pct50', 'B3_pct60', 'B3_pct70', 'B3_pct80', 'B3_pct90', 'B3_pct100', 'B3_mean', 'B3_std', 'B4_pct0', 'B4_pct10', 'B4_pct20', 'B4_pct30', 'B4_pct40', 'B4_pct50', 'B4_pct60', 'B4_pct70', 'B

In [5]:
X_train

,geolocation_name,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,B1_pct0,B1_pct10,B1_pct20,B1_pct30,B1_pct40,B1_pct50,B1_pct60,B1_pct70,B1_pct80,B1_pct90,B1_pct100,B1_mean,B1_std,B2_pct0,B2_pct10,B2_pct20,B2_pct30,B2_pct40,B2_pct50,B2_pct60,B2_pct70,B2_pct80,B2_pct90,B2_pct100,B2_mean,B2_std,B3_pct0,B3_pct10,B3_pct20,B3_pct30,B3_pct40,B3_pct50,B3_pct60,B3_pct70,B3_pct80,B3_pct90,B3_pct100,B3_mean,B3_std,B4_pct0,B4_pct10,B4_pct20,B4_pct30,B4_pct40,B4_pct50,B4_pct60,B4_pct70,B4_pct80,B4_pct90,B4_pct100,B4_mean,B4_std,B5_pct0,B5_pct10,B5_pct20,B5_pct30,B5_pct40,B5_pct50,B5_pct60,B5_pct70,B5_pct80,B5_pct90,B5_pct100,B5_mean,B5_std,B6_pct0,B6_pct10,B6_pct20,B6_pct30,B6_pct40,B6_pct50,B6_pct60,B6_pct70,B6_pct80,B6_pct90,B6_pct100,B6_mean,B6_std,B7_pct0,B7_pct10,B7_pct20,B7_pct30,...,BSI_pct50,BSI_pct60,BSI_pct70,BSI_pct80,BSI_pct90,BSI_pct100,BSI_mean,BSI_std,NBSI_pct0,NBSI_pct10,NBSI_pct20,NBSI_pct30,NBSI_pct40,NBSI_pct50,NBSI_pct60,NBSI_pct70,NBSI_pct80,NBSI_pct90,NBSI_pct100,NBSI_mean,NBSI_std,NDWI_pct0,NDWI_pct10,NDWI_pct20,NDWI_pct30,NDWI_pct40,NDWI_pct50,NDWI_pct60,NDWI_pct70,NDWI_pct80,NDWI_pct90,NDWI_pct100,NDWI_mean,NDWI_std,MNDWI_pct0,MNDWI_pct10,MNDWI_pct20,MNDWI_pct30,MNDWI_pct40,MNDWI_pct50,MNDWI_pct60,MNDWI_pct70,MNDWI_pct80,MNDWI_pct90,MNDWI_pct100,MNDWI_mean,MNDWI_std,Clay_pct0,Clay_pct10,Clay_pct20,Clay_pct30,Clay_pct40,Clay_pct50,Clay_pct60,Clay_pct70,Clay_pct80,Clay_pct90,Clay_pct100,Clay_mean,Clay_std,IronOxide_pct0,IronOxide_pct10,IronOxide_pct20,IronOxide_pct30,IronOxide_pct40,IronOxide_pct50,IronOxide_pct60,IronOxide_pct70,IronOxide_pct80,IronOxide_pct90,IronOxide_pct100,IronOxide_mean,IronOxide_std,Carbonate_pct0,Carbonate_pct10,Carbonate_pct20,Carbonate_pct30,Carbonate_pct40,Carbonate_pct50,Carbonate_pct60,Carbonate_pct70,Carbonate_pct80,Carbonate_pct90,Carbonate_pct100,Carbonate_mean,Carbonate_std,avg_rad_pct0,avg_rad_pct10,avg_rad_pct20,avg_rad_pct30,avg_rad_pct40,avg_rad_pct50,avg_rad_pct60,avg_rad_pct70,avg_rad_pct80,avg_rad_pct90,avg_rad_pct100,avg_rad_mean,avg_rad_std,quarter_num
0,Dinagat Islands,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,79.0,235.0,282.0,318.0,360.0,411.0,491.0,676.0,1505.0,4305.9,12442.0,1306.214859,2083.542109,109.0,283.0,318.0,354.0,396.0,448.0,526.0,702.0,1390.0,4237.0,12850.0,1304.793386,2028.163855,112.0,492.0,544.0,595.0,653.0,724.0,814.0,973.0,1501.0,4131.0,12429.0,1476.873568,1876.246837,55.0,236.0,272.0,319.0,380.0,463.0,568.0,775.0,1406.0,3918.0,12107.0,1239.978003,1876.147938,66.0,799.0,881.0,956.0,1042.0,1147.0,1267.0,1438.0,1853.0,4381.0,12849.0,1836.360197,1858.186597,45.0,2110.0,2415.0,2593.0,2738.0,2872.0,3015.0,3193.0,3482.0,4747.0,12033.0,3184.346665,1428.959278,77.0,2458.0,2910.0,3151.0,...,-0.259107,-0.195594,-0.138990,-0.096317,-0.051140,0.397685,-0.240089,0.152624,-0.696751,-0.452918,-0.427833,-0.402794,-0.366435,-0.313729,-0.255161,-0.200000,-0.147581,-0.084259,0.439789,-0.286398,0.153233,-0.850387,-0.750380,-0.726949,-0.701971,-0.670951,-0.633071,-0.586303,-0.486990,-0.261191,-0.037702,0.832858,-0.518670,0.267313,-0.738832,-0.507313,-0.477261,-0.452655,-0.428246,-0.401658,-0.367091,-0.283019,-0.096182,0.100802,0.863447,-0.303787,0.235211,0.607184,1.239295,1.393008,1.631015,1.863482,2.048593,2.204152,2.343933,2.459103,2.569671,3.558559,1.967181,0.498286,0.106383,0.747967,0.805690,0.853448,0.894484,0.928119,0.963603,1.018124,1.111111,1.268349,3.570909,0.981737,0.258615,0.169127,0.345718,0.368197,0.391037,0.426642,0.482034,0.549976,0.622843,0.696757,0.787294,2.256161,0.537562,0.201261,-0.04,0.11,0.13,0.15,0.16,0.17,0.18,0.20,0.22,0.27,7.700000,0.209104,0.311269,3
1,29000 Nara,Japan,2024,3.928801e+12,143.968241,Yes,Yes,High income,No,No,Yes

## クロスバリデーション実行

In [6]:
print('[LightGBM]')
lgb_oof, lgb_eval, lgb_score = run_cv(
    make_lgb_fn(LGB_PARAMS, CAT_COLS), X_train, y_train, X_eval,
    n_splits=N_SPLITS, seed=SEED,
)

print('\n[CatBoost]')
cb_oof, cb_eval, cb_score = run_cv(
    make_cb_fn(CB_PARAMS, CAT_COLS), X_train, y_train, X_eval,
    n_splits=N_SPLITS, seed=SEED,
)

[LightGBM]
[200]	valid_0's rmse: 203.184
  Fold 1 RMSLE: 0.2179
  Fold 2 RMSLE: 0.2393
  Fold 3 RMSLE: 0.2392
  OOF RMSLE: 0.2324 | Mean Fold: 0.2322

[CatBoost]
0:	learn: 800.5800735	test: 803.7879116	best: 803.7879116 (0)	total: 87.3ms	remaining: 1m 27s
200:	learn: 112.6228709	test: 210.3317911	best: 210.0128210 (188)	total: 2.79s	remaining: 11.1s
400:	learn: 67.2050753	test: 207.9452528	best: 207.9452528 (400)	total: 5.54s	remaining: 8.28s
600:	learn: 41.0935147	test: 207.2207662	best: 207.1747654 (598)	total: 8.1s	remaining: 5.38s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 207.154566
bestIteration = 602

Shrink model to first 603 iterations.
  Fold 1 RMSLE: 0.2418
0:	learn: 809.3073903	test: 785.7978370	best: 785.7978370 (0)	total: 13.9ms	remaining: 13.9s
200:	learn: 118.8380506	test: 161.5379801	best: 161.5333203 (199)	total: 2.58s	remaining: 10.3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 160.4014388
bestIteration = 235

Shrink model

## 結果サマリ

In [7]:
ensemble_oof   = (lgb_oof + cb_oof) / 2
ensemble_score = rmsle(y_train, ensemble_oof)

print('=' * 40)
print('モデル比較 (3-Fold CV)')
print('=' * 40)
print(f'LightGBM OOF RMSLE : {lgb_score:.4f}')
print(f'CatBoost OOF RMSLE : {cb_score:.4f}')
print(f'Ensemble OOF RMSLE : {ensemble_score:.4f}')

モデル比較 (3-Fold CV)
LightGBM OOF RMSLE : 0.2324
CatBoost OOF RMSLE : 0.2358
Ensemble OOF RMSLE : 0.2274


## 予測結果の保存

In [8]:
from src.io import save_oof

# OOF 予測を元テーブルにマージして保存
oof_df = save_oof(
    oof_preds={'lgb': lgb_oof, 'cb': cb_oof, 'ensemble': ensemble_oof},
    train_df=train_df,
    result_dir=RESULT_DIR,
)

# スコアの保存
pd.DataFrame({
    'model':     ['LightGBM', 'CatBoost', 'Ensemble'],
    'oof_rmsle': [lgb_score,  cb_score,   ensemble_score],
}).to_csv(f'{RESULT_DIR}/scores.csv', index=False)
print(f'Saved -> {RESULT_DIR}/scores.csv')

# 提出ファイルの保存（X_eval が揃っている場合のみ）
if lgb_eval is not None and cb_eval is not None:
    ensemble_eval = (lgb_eval + cb_eval) / 2
    save_submission(lgb_eval,     'submission_lgb.csv',      eval_df, RESULT_DIR)
    save_submission(cb_eval,      'submission_cb.csv',       eval_df, RESULT_DIR)
    ens_sub = save_submission(ensemble_eval, 'submission_ensemble.csv', eval_df, RESULT_DIR)
    display(ens_sub.head())
else:
    print('X_eval=None のため提出ファイルの保存をスキップしました。')

oof_df.head()

Saved -> ../data/result/case002/oof_predictions.csv
Saved -> ../data/result/case002/scores.csv
Saved -> ../data/result/case002/submission_lgb.csv
Saved -> ../data/result/case002/submission_cb.csv
Saved -> ../data/result/case002/submission_ensemble.csv


,data_id,construction_cost_per_m2_usd
0,3TOW4,1829.041771
1,493WX,1686.730984
2,UYP04,1853.867522
3,FN33V,214.548366
4,CPRHV,1722.765096


,data_id,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,sentinel2_tiff_file_name,viirs_tiff_file_name,construction_cost_per_m2_usd,B1_pct0,B1_pct10,B1_pct20,B1_pct30,B1_pct40,B1_pct50,B1_pct60,B1_pct70,B1_pct80,B1_pct90,B1_pct100,B1_mean,B1_std,B2_pct0,B2_pct10,B2_pct20,B2_pct30,B2_pct40,B2_pct50,B2_pct60,B2_pct70,B2_pct80,B2_pct90,B2_pct100,B2_mean,B2_std,B3_pct0,B3_pct10,B3_pct20,B3_pct30,B3_pct40,B3_pct50,B3_pct60,B3_pct70,B3_pct80,B3_pct90,B3_pct100,B3_mean,B3_std,B4_pct0,B4_pct10,B4_pct20,B4_pct30,B4_pct40,B4_pct50,B4_pct60,B4_pct70,B4_pct80,B4_pct90,B4_pct100,B4_mean,B4_std,B5_pct0,B5_pct10,B5_pct20,B5_pct30,B5_pct40,B5_pct50,B5_pct60,B5_pct70,B5_pct80,B5_pct90,B5_pct100,B5_mean,B5_std,B6_pct0,B6_pct10,B6_pct20,B6_pct30,B6_pct40,B6_pct50,B6_pct60,B6_pct70,B6_pct80,B6_pct90,B6_pct100,B6_mean,...,BSI_pct70,BSI_pct80,BSI_pct90,BSI_pct100,BSI_mean,BSI_std,NBSI_pct0,NBSI_pct10,NBSI_pct20,NBSI_pct30,NBSI_pct40,NBSI_pct50,NBSI_pct60,NBSI_pct70,NBSI_pct80,NBSI_pct90,NBSI_pct100,NBSI_mean,NBSI_std,NDWI_pct0,NDWI_pct10,NDWI_pct20,NDWI_pct30,NDWI_pct40,NDWI_pct50,NDWI_pct60,NDWI_pct70,NDWI_pct80,NDWI_pct90,NDWI_pct100,NDWI_mean,NDWI_std,MNDWI_pct0,MNDWI_pct10,MNDWI_pct20,MNDWI_pct30,MNDWI_pct40,MNDWI_pct50,MNDWI_pct60,MNDWI_pct70,MNDWI_pct80,MNDWI_pct90,MNDWI_pct100,MNDWI_mean,MNDWI_std,Clay_pct0,Clay_pct10,Clay_pct20,Clay_pct30,Clay_pct40,Clay_pct50,Clay_pct60,Clay_pct70,Clay_pct80,Clay_pct90,Clay_pct100,Clay_mean,Clay_std,IronOxide_pct0,IronOxide_pct10,IronOxide_pct20,IronOxide_pct30,IronOxide_pct40,IronOxide_pct50,IronOxide_pct60,IronOxide_pct70,IronOxide_pct80,IronOxide_pct90,IronOxide_pct100,IronOxide_mean,IronOxide_std,Carbonate_pct0,Carbonate_pct10,Carbonate_pct20,Carbonate_pct30,Carbonate_pct40,Carbonate_pct50,Carbonate_pct60,Carbonate_pct70,Carbonate_pct80,Carbonate_pct90,Carbonate_pct100,Carbonate_mean,Carbonate_std,avg_rad_pct0,avg_rad_pct10,avg_rad_pct20,avg_rad_pct30,avg_rad_pct40,avg_rad_pct50,avg_rad_pct60,avg_rad_pct70,avg_rad_pct80,avg_rad_pct90,avg_rad_pct100,avg_rad_mean,avg_rad_std,oof_lgb,oof_cb,oof_ensemble
0,LP81L,Dinagat Islands,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,sentinel_2_dinagat_islands_2019-Q3.tif,viirs_dinagat_islands_2019-Q3.tif,129.997420,79.0,235.0,282.0,318.0,360.0,411.0,491.0,676.0,1505.0,4305.9,12442.0,1306.214859,2083.542109,109.0,283.0,318.0,354.0,396.0,448.0,526.0,702.0,1390.0,4237.0,12850.0,1304.793386,2028.163855,112.0,492.0,544.0,595.0,653.0,724.0,814.0,973.0,1501.0,4131.0,12429.0,1476.873568,1876.246837,55.0,236.0,272.0,319.0,380.0,463.0,568.0,775.0,1406.0,3918.0,12107.0,1239.978003,1876.147938,66.0,799.0,881.0,956.0,1042.0,1147.0,1267.0,1438.0,1853.0,4381.0,12849.0,1836.360197,1858.186597,45.0,2110.0,2415.0,2593.0,2738.0,2872.0,3015.0,3193.0,3482.0,4747.0,12033.0,3184.346665,...,-0.138990,-0.096317,-0.051140,0.397685,-0.240089,0.152624,-0.696751,-0.452918,-0.427833,-0.402794,-0.366435,-0.313729,-0.255161,-0.200000,-0.147581,-0.084259,0.439789,-0.286398,0.153233,-0.850387,-0.750380,-0.726949,-0.701971,-0.670951,-0.633071,-0.586303,-0.486990,-0.261191,-0.037702,0.832858,-0.518670,0.267313,-0.738832,-0.507313,-0.477261,-0.452655,-0.428246,-0.401658,-0.367091,-0.283019,-0.096182,0.100802,0.863447,-0.303787,0.235211,0.607184,1.239295,1.393008,1.631015,1.863482,2.048593,2.204152,2.343933,2.459103,2.569671,3.558559,1.967181,0.498286,0.106383,0.747967,0.805690,0.853448,0.894484,0.928119,0.963603,1.018124,1.111111,1.268349,3.570909,0.981737,0.258615,0.169127,0.345718,0.368197,0.391037,0.426642,0.482034,0.549976,0.622843,0.696757,0.787294,2.256161,0.537562,0.201261,-0.04,0.11,0.13,0.15,0.16,0.17,0.18,0.20,0.22,0.27,7.700000,0.2091